# 05 Random forest voxel classifier

Each voxel is described by 9 handcrafted features (`rf_features` in `src/utils.py`) by intensity, intensity smoothed at 3 scales, gradient magnitude at 2 scales and position in the volume.
Go see random forest here if you dont know how it works :) : https://youtu.be/gkXX4h3qYm4?si=RKuJzOCTtZBzoPwT
- Only voxels inside a box around the training LA positions are classified (same box as the baseline). This makes prediction much faster.
- For training, the same number of LA and background voxels is sampled in each training volume.
- Prediction is probability ≥ threshold, then only the largest connected component is kept.

Because the training set is balanced, the predicted probabilities are too high for the real class proportions, and a threshold of 0.5 gives too many LA voxels. The threshold is therefore chosen with the maximum tree depth by 5-fold cross-validation on the development set (seed 0).

In [1]:
import json
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# shared functions of the project (src/utils.py)
sys.path.insert(0, "../src")
from utils import (FEATURE_NAMES, RESULTS, SPLITS, evaluate, fold_cases, keep_largest_component, la_prior,
                   load_case, normalize, rf_segment, sample_training_set)

N_PER_CLASS = 5000  # voxels sampled per class and per training volume
N_TREES = 100  # number of trees in the forest
# values tried in cross-validation
DEPTHS = [10, 20, None]  # maximum depth of the trees (None = no limit)
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]  # probability above which a voxel is called LA

# load the 15 development volumes once, with normalised intensities
data = {}
for c in SPLITS["dev"]:
    x, y, spacing = load_case(c)
    data[c] = (normalize(x), y, spacing)

## Cross-validation

In [2]:
rows, importances = [], []
for k in range(len(SPLITS["folds"])):
    # 12 volumes to train, 3 volumes to evaluate
    train, val = fold_cases(k)
    _, box = la_prior([data[c][1] for c in train])
    # training voxels: 5000 LA + 5000 background per training volume
    X, Y = sample_training_set([data[c][:2] for c in train], box, N_PER_CLASS, seed=0)
    for depth in DEPTHS:
        rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=depth, n_jobs=-1, random_state=0).fit(X, Y)
        importances.append(rf.feature_importances_)
        for c in val:
            x, y, spacing = data[c]
            # LA probability of every voxel of the box (computed once, then thresholded several times)
            _, prob = rf_segment(rf, x, box)
            for t in THRESHOLDS:
                pred = keep_largest_component(prob >= t)
                rows.append({"fold": k, "case": c, "max_depth": str(depth), "threshold": t, **evaluate(pred, y, spacing)})
    print(f"fold {k} done")

# one row per (volume, setting): saved, then averaged per setting
cv = pd.DataFrame(rows)
cv.to_csv(RESULTS / "cv_random_forest.csv", index=False)
cv.groupby(["max_depth", "threshold"])[["dice", "hd95_mm", "sensitivity", "precision", "volume_error_ml"]].mean().round(3)

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done


dice  hd95_mm  sensitivity  precision  volume_error_ml
max_depth threshold                                                         
10        0.50       0.533   29.291        0.928      0.381          147.587
          0.60       0.564   27.604        0.907      0.417          121.018
          0.70       0.602   25.872        0.880      0.468           92.052
          0.80       0.640   23.408        0.841      0.529           61.696
          0.90       0.677   20.143        0.748      0.637           18.752
          0.95       0.616   18.425        0.530      0.778          -32.066
20        0.50       0.590   25.181        0.897      0.448          102.501
          0.60       0.624   23.415        0.873      0.497           79.061
          0.70       0.651   21.934        0.842      0.545           57.353
          0.80       0.675   20.174        0.799      0.601           34.622
          0.90       0.691   17.382        0.721      0.685            6.128
          0.95       0.673   16.080        0.625      0.761          -17.488
None      0.50       0.600   24.611        0.887      0.461           94.350
          0.60       0.636   22.910        0.859      0.517           68.936
          0.70       0.663   21.409        0.822      0.571           46.431
          0.80       0.684   19.175        0.769      0.635           22.599
          0.90       0.684   16.647        0.677      0.719           -5.196
          0.95       0.657   15.552        0.589      0.781          -24.308

In [3]:
# chosen configuration = best mean Dice over the 15 validation volumes
best_depth, best_t = cv.groupby(["max_depth", "threshold"])["dice"].mean().idxmax()
print(f"chosen: max_depth {best_depth}, threshold {best_t}")

# results of the chosen setting for each volume
cv[(cv["max_depth"] == best_depth) & (cv["threshold"] == best_t)].round(3)

chosen: max_depth 20, threshold 0.9


,fold,case,max_depth,threshold,dice,hd95_mm,sensitivity,precision,volume_ml,ref_volume_ml,volume_error_ml
22,0,la_003,20,0.9,0.679,27.628,0.804,0.588,131.713,96.335,35.378
28,0,la_010,20,0.9,0.477,25.031,0.668,0.371,145.211,80.526,64.685
34,0,la_023,20,0.9,0.777,16.685,0.835,0.727,106.239,92.484,13.756
82,1,la_005,20,0.9,0.741,12.182,0.680,0.813,104.231,124.608,-20.377
88,1,la_017,20,0.9,0.409,24.968,0.310,0.598,41.941,80.871,-38.929
94,1,la_022,20,0.9,0.775,15.727,0.944,0.657,103.236,71.820,31.416
100,1,la_026,20,0.9,0.753,12.070,0.717,0.793,104.231,115.305,-11.073
148,2,la_007,20,0.9,0.814,12.979,0.848,0.783,128.555,118.685,9.870
154,2,la_014,20,0.9,0.787,12.797,0.698,0.903,112.747,145.879,-33.133
160,2,la_024,20,0.9,0.689,16.725,0.709,0.669,105.385,99.518,5.867


In [4]:
# feature importance: how much each feature is used by the trees (mean over folds and depths)
pd.Series(np.mean(importances, axis=0), index=FEATURE_NAMES).sort_values(ascending=False).round(3)

smooth_1      0.247
smooth_2      0.207
pos_0         0.174
smooth_4      0.125
pos_1         0.102
intensity     0.080
pos_2         0.038
gradient_2    0.015
gradient_1    0.013
dtype: float64

In [5]:
# save the chosen configuration (used by the final training and test notebooks)
config_file = RESULTS / "chosen_configs.json"
configs = json.loads(config_file.read_text()) if config_file.exists() else {}
configs["random_forest"] = {"max_depth": None if best_depth == "None" else int(best_depth),
                            "threshold": float(best_t), "n_trees": N_TREES, "n_per_class": N_PER_CLASS}
config_file.write_text(json.dumps(configs, indent=2))

205